## Importing Packages

In [18]:
import pandas as pd
import numpy as np
import polars as pl
import glob

import matplotlib.pyplot as plt
import seaborn as sns
import os

In [19]:
!rm -r tvDatafeed


In [20]:
import os
os.makedirs("tvDatafeed", exist_ok=True)


In [21]:
open("tvDatafeed/__init__.py", "w").close()


In [57]:
code = """
import pandas as pd
import warnings
from enum import Enum
from datetime import datetime, timedelta

try:
    import yfinance as yf
except:
    yf = None

class Interval(Enum):
    in_1_minute = "1m"
    in_3_minute = "3m"
    in_5_minute = "5m"
    in_15_minute = "15m"
    in_30_minute = "30m"
    in_60_minute = "60m"
    in_daily = "1d"
    in_1_day = "1d"
    in_weekly = "1wk"
    in_monthly = "1mo"

def _map_interval(interval):
    if isinstance(interval, Interval):
        return interval.value
    return str(interval)

def _to_yf_symbol(symbol, exchange):
    symbol = symbol.upper()
    exchange = exchange.upper()
    if exchange == "NSE":
        return symbol + ".NS"
    if exchange == "BSE":
        return symbol + ".BO"
    return symbol

class TvDatafeed:
    def __init__(self, username=None, password=None):
        if yf is None:
            raise ImportError("yfinance is required but not installed.")
        self.username = username
        self.password = password

    def get_hist(self, symbol, exchange="NSE", interval=Interval.in_daily, n_bars=1000, fut=False):
        yf_symbol = _to_yf_symbol(symbol, exchange)
        yf_interval = _map_interval(interval)

        # map n_bars to period (yfinance requires period for minute data)
        if yf_interval.endswith("m"):
            period = "60d"    # max allowed period for intraday
        else:
            years = max(1, int(n_bars / 250))
            period = f"{years}y"

        try:
            df = yf.download(
                tickers=yf_symbol,
                period=period,
                interval=yf_interval,
                progress=False,
                auto_adjust=True # Explicitly set auto_adjust to True to prevent MultiIndex columns
            )
        except Exception as e:
            raise RuntimeError(f"Data fetch failed: {e}")

        if df is None or df.empty:
            raise RuntimeError(f"No data returned for {symbol} from yfinance.")

        df.index.name = "date"
        # Convert column names to lowercase and flatten if they are MultiIndex tuples
        df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower() for c in df.columns]

        # Ensure all required OHLCV columns exist and handle missing ones
        keep = ["open", "high", "low", "close", "volume"]
        for k in keep:
            if k not in df.columns:
                df[k] = None # Fill with None if a column is genuinely missing (unlikely for OHLCV from yfinance)

        # Tail n_bars
        df = df[keep].tail(n_bars)
        return df

"""

with open("tvDatafeed/__init__.py", "w") as f:
    f.write(code)

print("tvDatafeed installed!")

tvDatafeed installed!


In [58]:
import sys
sys.modules.pop('tvDatafeed', None)

<module 'tvDatafeed' from '/content/tvDatafeed/__init__.py'>

In [59]:
from tvDatafeed import TvDatafeed, Interval
print("SUCCESS!", TvDatafeed, Interval.in_daily)

SUCCESS! <class 'tvDatafeed.TvDatafeed'> Interval.in_daily


## Creating the Dataset

### Logging into TVFeed

In [60]:
tv = TvDatafeed(username = 'jaganathapandiyan12', password = 'PASS$1234TO5678')

### Downloading the stocks in NIFTY 500 (list downloaded from NSE website on 16th November)

In [26]:
data_nifty500 = pd.read_csv('ind_nifty500list.csv')

symbols_nifty500 = data_nifty500['Symbol'].to_list()
symbols_nifty500 = [s.replace("-", "_") for s in symbols_nifty500]
symbols_nifty500.remove('DUMMYSKFIN')


print(f'Total stocks: {len(symbols_nifty500)}')

Total stocks: 501


In [27]:
import pandas as pd

df = pd.read_csv("ind_nifty500list.csv")

symbols_raw = df['Symbol'].astype(str).str.strip().tolist()

symbols_clean = []
for s in symbols_raw:
    s = s.upper().strip()

    # Skip empty values or invalid
    if not s or s == 'nan':
        continue

    # Prevent double .NS
    if s.endswith(".NS"):
        symbols_clean.append(s)
    else:
        symbols_clean.append(s + ".NS")

print("Sample cleaned symbols:", symbols_clean[:10])


Sample cleaned symbols: ['360ONE.NS', '3MINDIA.NS', 'ABB.NS', 'ACC.NS', 'ACMESOLAR.NS', 'AIAENG.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'AWL.NS', 'AADHARHFC.NS']


In [61]:
failed = []

import os, time
os.makedirs("data/raw/prices", exist_ok=True)

for symbol in symbols_clean:
    try:
        df = tv.get_hist(
            symbol=symbol.replace(".NS","",1), # Use replace count=1 to avoid issues if .NS appears elsewhere
            exchange='NSE',
            interval=Interval.in_daily,
            n_bars=2000
        )

        if df is None or df.empty:
            raise ValueError(f"No data returned for {symbol}")

        outname = symbol.replace(".NS", "",1)
        df.to_parquet(f"data/raw/prices/{outname}.parquet")

        print("✔ Downloaded:", symbol)

    except Exception as e:
        print("✖ Failed:", symbol, "→", e)
        failed.append(symbol)

    time.sleep(0.05)

✔ Downloaded: 360ONE.NS
✔ Downloaded: 3MINDIA.NS
✔ Downloaded: ABB.NS
✔ Downloaded: ACC.NS
✔ Downloaded: ACMESOLAR.NS
✔ Downloaded: AIAENG.NS
✔ Downloaded: APLAPOLLO.NS
✔ Downloaded: AUBANK.NS
✔ Downloaded: AWL.NS
✔ Downloaded: AADHARHFC.NS
✔ Downloaded: AARTIIND.NS
✔ Downloaded: AAVAS.NS
✔ Downloaded: ABBOTINDIA.NS
✔ Downloaded: ACE.NS
✔ Downloaded: ADANIENSOL.NS
✔ Downloaded: ADANIENT.NS
✔ Downloaded: ADANIGREEN.NS
✔ Downloaded: ADANIPORTS.NS
✔ Downloaded: ADANIPOWER.NS
✔ Downloaded: ATGL.NS
✔ Downloaded: ABCAPITAL.NS
✔ Downloaded: ABFRL.NS
✔ Downloaded: ABLBL.NS
✔ Downloaded: ABREL.NS
✔ Downloaded: ABSLAMC.NS
✔ Downloaded: ADVENTHTL.NS
✔ Downloaded: AEGISLOG.NS
✔ Downloaded: AEGISVOPAK.NS
✔ Downloaded: AFCONS.NS
✔ Downloaded: AFFLE.NS
✔ Downloaded: AJANTPHARM.NS
✔ Downloaded: AKUMS.NS
✔ Downloaded: AKZOINDIA.NS
✔ Downloaded: APLLTD.NS
✔ Downloaded: ALKEM.NS
✔ Downloaded: ALKYLAMINE.NS
✔ Downloaded: ALOKINDS.NS
✔ Downloaded: ARE&M.NS
✔ Downloaded: AMBER.NS
✔ Downloaded: AMBUJACEM.NS


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['DUMMYSKFIN.NS']: YFPricesMissingError('possibly delisted; no price data found  (period=8y) (Yahoo error = "No data found, symbol may be delisted")')


✔ Downloaded: DRREDDY.NS
✖ Failed: DUMMYSKFIN.NS → No data returned for DUMMYSKFIN from yfinance.
✔ Downloaded: EIDPARRY.NS
✔ Downloaded: EIHOTEL.NS
✔ Downloaded: EICHERMOT.NS
✔ Downloaded: ELECON.NS
✔ Downloaded: ELGIEQUIP.NS
✔ Downloaded: EMAMILTD.NS
✔ Downloaded: EMCURE.NS
✔ Downloaded: ENDURANCE.NS
✔ Downloaded: ENGINERSIN.NS
✔ Downloaded: ERIS.NS
✔ Downloaded: ESCORTS.NS
✔ Downloaded: ETERNAL.NS
✔ Downloaded: EXIDEIND.NS
✔ Downloaded: NYKAA.NS
✔ Downloaded: FEDERALBNK.NS
✔ Downloaded: FACT.NS
✔ Downloaded: FINCABLES.NS
✔ Downloaded: FINPIPE.NS
✔ Downloaded: FSL.NS
✔ Downloaded: FIVESTAR.NS
✔ Downloaded: FORCEMOT.NS
✔ Downloaded: FORTIS.NS
✔ Downloaded: GAIL.NS
✔ Downloaded: GVT&D.NS
✔ Downloaded: GMRAIRPORT.NS
✔ Downloaded: GRSE.NS
✔ Downloaded: GICRE.NS
✔ Downloaded: GILLETTE.NS
✔ Downloaded: GLAND.NS
✔ Downloaded: GLAXO.NS
✔ Downloaded: GLENMARK.NS
✔ Downloaded: MEDANTA.NS
✔ Downloaded: GODIGIT.NS
✔ Downloaded: GPIL.NS
✔ Downloaded: GODFRYPHLP.NS
✔ Downloaded: GODREJAGRO.NS
✔ Do

In [38]:
import glob
import pandas as pd

all_files = glob.glob("data/raw/prices/*.parquet")
dfs = []

for f in all_files:
    symbol = f.split("/")[-1].replace(".parquet","")
    temp = pd.read_parquet(f)
    temp["symbol"] = symbol
    dfs.append(temp)

df_all = pd.concat(dfs)
df_all.to_parquet("data/processed/Prices_all.parquet")

print("Processed file created:", "data/processed/Prices_all.parquet")


Processed file created: data/processed/Prices_all.parquet


In [12]:
!git clone --branch Rokith_algo https://github.com/jagan010101/Algo-Trading.git


Cloning into 'Algo-Trading'...
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Merging the files into a single file for analysis

In [62]:
import pandas as pd
import polars as pl
import os
import glob
import pyarrow # pandas will use it if available for to_parquet engine

RAW_DIR = "data/raw/prices"
OUT_FILE = "data/Processed/Prices_all.parquet"

def merge_parquet_files():
    # Remove the existing file if it's corrupted or incomplete
    if os.path.exists(OUT_FILE):
        os.remove(OUT_FILE)
        print(f"Removed existing corrupted file: {OUT_FILE}")

    files = glob.glob(os.path.join(RAW_DIR, "*.parquet"))

    if len(files) == 0:
        print("No parquet files found in data/raw/prices/")
        return

    dfs = []

    for f in files:
        symbol = os.path.basename(f).replace(".parquet", "")
        try:
            df_polars = (pl.read_parquet(f).with_columns([pl.lit(symbol).alias("symbol"),
                                                          pl.col("date").cast(pl.Date).alias("date")
                                                         ]))
            dfs.append(df_polars)
        except Exception as e:
            print(f"Error reading individual parquet file {f}: {e}")
            continue

    if not dfs:
        print("No valid dataframes to merge.")
        return

    final_df_polars = pl.concat(dfs, how="vertical")
    final_df_polars = final_df_polars.sort(["symbol", "date"])

    os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)

    try:
        # Convert to pandas DataFrame and write using pandas' to_parquet
        # This will use pyarrow or fastparquet engine if available.
        # pyarrow is usually installed with polars environments.
        final_df_pandas = final_df_polars.to_pandas()
        final_df_pandas.to_parquet(OUT_FILE, engine='pyarrow', compression='snappy')

        print(f"Master datafile created using pandas.to_parquet: {OUT_FILE}")
        print(f"Rows: {final_df_polars.height}, Columns: {final_df_polars.width}")

        # Verification step: Try to read the file immediately after writing with Polars
        try:
            _ = pl.read_parquet(OUT_FILE)
            print(f"Verification successful: {OUT_FILE} can be read back by Polars.")
        except Exception as e:
            print(f"Verification failed for {OUT_FILE} with Polars read, even after pandas write: {e}")
            if os.path.exists(OUT_FILE):
                os.remove(OUT_FILE)
            print(f"Removed problematic file: {OUT_FILE}")
            return

    except Exception as e:
        print(f"Error writing parquet file {OUT_FILE} using pandas.to_parquet: {e}")
        if os.path.exists(OUT_FILE):
            os.remove(OUT_FILE)
        return


if __name__ == "__main__":
    merge_parquet_files()

Removed existing corrupted file: data/Processed/Prices_all.parquet
Master datafile created using pandas.to_parquet: data/Processed/Prices_all.parquet
Rows: 832458, Columns: 7
Verification successful: data/Processed/Prices_all.parquet can be read back by Polars.


## Calculating the Required Ratios

In [63]:
df = pl.read_parquet("data/Processed/Prices_all.parquet", use_pyarrow=True)
df = df.sort(["symbol", "date"])

### 1) MACD (12-day EMA - 26-day EMA)

In [90]:
df = df.with_columns([
    pl.col("close").ewm_mean(span = 12).over("symbol").alias("ema_12"),
    pl.col("close").ewm_mean(span = 26).over("symbol").alias("ema_26"),]).with_columns([
    (pl.col("ema_12") - pl.col("ema_26")).alias("macd")])

### 2) 14-day ROC

In [91]:
df = df.with_columns([
    ((pl.col("close") - pl.col("close").shift(14))
     / pl.col("close").shift(14)).over("symbol").alias("roc_14")])

### 3) 14-day ADX

In [92]:
df = df.with_columns([
    pl.col("high").shift(1).over("symbol").alias("prev_high"),
    pl.col("low").shift(1).over("symbol").alias("prev_low"),
    pl.col("close").shift(1).over("symbol").alias("prev_close"),])

df = df.with_columns([
    (pl.col("high") - pl.col("low")).alias("hl_range"),
    (pl.col("high") - pl.col("prev_close")).abs().alias("hc_range"),
    (pl.col("low") - pl.col("prev_close")).abs().alias("lc_range"),])

df = df.with_columns([
    pl.max_horizontal([
        pl.col("hl_range"),
        pl.col("hc_range"),
        pl.col("lc_range"),
    ]).alias("tr")])

df = df.with_columns([
    (pl.col("high") - pl.col("prev_high")).alias("up_move"),
    (pl.col("prev_low") - pl.col("low")).alias("down_move"),])

df = df.with_columns([
    pl.when(
        (pl.col("up_move") > pl.col("down_move")) & (pl.col("up_move") > 0)
        ).then(pl.col("up_move")).otherwise(0).alias("plus_dm"),

    pl.when(
        (pl.col("down_move") > pl.col("up_move")) & (pl.col("down_move") > 0)
        ).then(pl.col("down_move")).otherwise(0).alias("minus_dm"),])

df = df.with_columns([
    pl.col("tr").rolling_mean(14).over("symbol").alias("atr_14"),
    pl.col("plus_dm").rolling_mean(14).over("symbol").alias("plus_dm_14"),
    pl.col("minus_dm").rolling_mean(14).over("symbol").alias("minus_dm_14"),])


df = df.with_columns([
    (100 * pl.col("plus_dm_14") / pl.col("atr_14")).alias("plus_di_14"),
    (100 * pl.col("minus_dm_14") / pl.col("atr_14")).alias("minus_di_14"),])


df = df.with_columns([
    (100 * (pl.col("plus_di_14") - pl.col("minus_di_14")).abs()
     / (pl.col("plus_di_14") + pl.col("minus_di_14"))
    ).alias("dx_14")])

df = df.with_columns([
    pl.col("dx_14").rolling_mean(14).over("symbol").alias("adx_14")])

### 4) 5-day VWAP

In [93]:
df = df.with_columns(
    ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("typical_price"))

df = df.with_columns([
    (pl.col("typical_price") * pl.col("volume")).rolling_sum(5).over("symbol").alias("tp_vol_sum_5"),
    pl.col("volume").rolling_sum(5).over("symbol").alias("vol_sum_5"),
                    ]).with_columns([(pl.col("tp_vol_sum_5") / pl.col("vol_sum_5")).alias("vwap_5")])

### 5) 14-day RSI

In [94]:
df = df.with_columns(
    pl.col("close").diff().over("symbol").alias("delta"))

df = df.with_columns([
    pl.col("delta").clip(lower_bound=0).alias("gain"),
    (-pl.col("delta").clip(upper_bound=0)).alias("loss")])

df = df.with_columns([
    pl.col("gain").rolling_mean(14).over("symbol").alias("avg_gain_14"),
    pl.col("loss").rolling_mean(14).over("symbol").alias("avg_loss_14")])

df = df.with_columns((
    100 - 100 / (1 + (pl.col("avg_gain_14") / pl.col("avg_loss_14")))).alias("rsi_14"))

### 6) 20-day Volume

In [95]:
df = df.with_columns(
    pl.col("volume").rolling_mean(20).over("symbol").alias("sma_vol_20"))

### 7) 14-day ATR

In [96]:
df = df.with_columns([
    pl.col("close").shift(1).over("symbol").alias("prev_close")])

df = df.with_columns([
    pl.max_horizontal([pl.col("high") - pl.col("low"),
                       (pl.col("high") - pl.col("prev_close")).abs(),
                       (pl.col("low")  - pl.col("prev_close")).abs()]).alias("true_range")])

df = df.with_columns([
    pl.col("true_range").rolling_mean(14).over("symbol").alias("atr_14")])

8) Daily Return

In [97]:
df = df.with_columns(
    (
        pl.col("close") / pl.col("close").shift(1) - 1
    )
    .over("symbol")
    .alias("daily_ret")
)

9) 60 Day Histotical volatility

In [98]:
df = df.with_columns(
    (
        pl.col("daily_ret")
        .rolling_std(60)
        .over("symbol")
        * (252 ** 0.5)
    )
    .alias("hv60")
)

In [105]:
# ============================
# NEW FEATURES FOR 3 STRATEGIES
# ============================

# # Peer RSI (Sector / Industry)
# df = df.with_columns(
#     pl.col("rsi_14").mean().over("sector").alias("avg_peer_rsi14")
# )

# MACD Signal Line
df = df.with_columns(
    pl.col("macd").ewm_mean(span=9).over("symbol").alias("macd_signal")
)

# VWAP proxy + VWAP 5-day
df = df.with_columns(
    ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("vwap")
)
df = df.with_columns(
    pl.col("vwap").rolling_mean(5).over("symbol").alias("vwap_5")
)

# Price above VWAP
df = df.with_columns(
    (pl.col("close") > pl.col("vwap")).alias("price_above_vwap")
)

# Price above VWAP 2-day streak
df = df.with_columns(
    (pl.col("price_above_vwap").cast(pl.Int8)
     .rolling_sum(2).over("symbol") >= 2)
    .alias("price_above_vwap2d")
)

# RSI Rising 1-day & 2-day streak
df = df.with_columns(
    (pl.col("rsi_14") > pl.col("rsi_14").shift(1).over("symbol"))
    .alias("rsi_rising")
)
df = df.with_columns(
    (pl.col("rsi_rising").cast(pl.Int8)
     .rolling_sum(2).over("symbol") >= 2)
    .alias("rsi_rising_2d")
)

# Volume spike above SMA20
df = df.with_columns(
    (pl.col("volume") > pl.col("sma_vol_20")).alias("vol_above_sma20")
)

# ROC 7
df = df.with_columns(
    ((pl.col("close") / pl.col("close").shift(7).over("symbol")) - 1)
    .alias("roc_7")
)

# Daily returns
df = df.with_columns(
    (pl.col("close") / pl.col("close").shift(1).over("symbol") - 1)
    .alias("daily_ret")
)


In [110]:
import polars as pl
import pandas as pd
import numpy as np



def compute_avg_peer_rsi(pl_df: pl.DataFrame, top_n: int = 10) -> pl.DataFrame:
    """
    Computes avg_peer_rsi14 for each symbol-date based on top-N correlated peers.

    Steps:
        1. Take last 500 days of data only (faster)
        2. Build pivot: date × symbol of daily_ret
        3. Compute correlation matrix between symbols
        4. Select top-N correlated peers for each symbol
        5. Build RSI pivot: date × symbol
        6. Average RSI values of peers for each date
        7. Merge back into main Polars DataFrame
    """

    # Convert to pandas for pivot + correlation (much faster)
    pdf = pl_df.to_pandas().sort_values(["symbol", "date"])

    # Limit to last 500 days for correlation calculation
    last_date = pdf["date"].max()
    cutoff = last_date - pd.Timedelta(days=500)
    pdf_recent = pdf[pdf["date"] >= cutoff].copy()

    # Pivot returns
    ret_pivot = (
        pdf_recent.pivot(index="date", columns="symbol", values="daily_ret")
                 .fillna(0.0)
    )

    # Correlation matrix
    corr = ret_pivot.corr().fillna(0.0)

    # Determine top-N correlated peers per symbol
    peer_map = {}
    for sym in corr.columns:
        peers = (
            corr[sym]
            .drop(labels=[sym], errors="ignore")
            .abs()
            .sort_values(ascending=False)
            .head(top_n)
            .index
            .tolist()
        )
        peer_map[sym] = peers

    # Build RSI pivot
    rsi_pivot = (
        pdf_recent.pivot(index="date", columns="symbol", values="rsi_14")
                 .fillna(50.0)
    )

    # Compute avg peer RSI time series for each symbol
    out = []

    for sym, peers in peer_map.items():
        if len(peers) == 0:
            continue

        # peer-average RSI for each date
        peer_series = rsi_pivot[peers].mean(axis=1)
        peer_series = peer_series.rename("avg_peer_rsi14")

        # assemble dataframe
        temp = pd.DataFrame({
            "date": rsi_pivot.index,
            "symbol": sym,
            "avg_peer_rsi14": peer_series.values
        })
        out.append(temp)

    # If no peers found at all, fallback
    if len(out) == 0:
        return pl_df.with_columns(pl.lit(50.0).alias("avg_peer_rsi14"))

    # Merge all peer results
    peer_df = pd.concat(out, ignore_index=True)

    # Merge back into original full dataset
    pdf = pdf.merge(peer_df, on=["date", "symbol"], how="left")

    # Fallback: if missing, use rolling mean of own RSI
    pdf["avg_peer_rsi14"] = pdf["avg_peer_rsi14"].fillna(
        pdf.groupby("symbol")["rsi_14"].transform(
            lambda s: s.rolling(20, min_periods=1).mean()
        )
    )

    # Back to Polars
    return pl.from_pandas(pdf)
df = compute_avg_peer_rsi(df)


### Dropping unnecessary columns and removing NULL rows

In [111]:
df = df.drop([
    "ema_12", "ema_26", "delta", "gain", "loss", "avg_gain_14", "avg_loss_14", "typical_price",
    "tp_vol_sum_5", "vol_sum_5", "prev_close", "true_range", "prev_high", "prev_low",
    "hl_range", "hc_range", "lc_range", "up_move", "down_move", "plus_dm", "minus_dm", "plus_dm_14",
    "minus_dm_14", "dx_14", "tr", "plus_di_14", "minus_di_14"
], strict=False)

In [112]:
print(f'Before dropping nulls: {df.shape}')

df = df.drop_nulls()

print(f'After dropping nulls: {df.shape}')

Before dropping nulls: (746426, 26)
After dropping nulls: (740388, 26)


In [113]:
df.head()

open,high,low,close,volume,date,symbol,atr_14,adx_14,vwap_5,rsi_14,sma_vol_20,macd,daily_ret,hv60,forward_return_5d,roc_14,macd_signal,vwap,price_above_vwap,price_above_vwap2d,rsi_rising,rsi_rising_2d,vol_above_sma20,roc_7,avg_peer_rsi14
f64,f64,f64,f64,i64,datetime[ms],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,f64,f64
182.580076,183.638205,180.453433,182.26886,13940,2020-05-07 00:00:00,"""360ONE""",12.822099,13.032697,189.7498,43.279746,59104.4,-10.033373,-0.004927,0.761586,-0.053785,-0.058666,-11.066269,182.120166,true,false,true,false,false,-0.002385,43.279746
177.465726,188.804366,177.465726,181.687897,8712,2020-05-08 00:00:00,"""360ONE""",12.817653,11.543867,187.246234,38.838752,39079.4,-9.844536,-0.003187,0.761099,-0.030947,-0.088809,-10.784042,182.652663,false,false,false,false,false,-0.025593,41.059249
179.05295,187.974476,177.922196,178.689865,9180,2020-05-11 00:00:00,"""360ONE""",12.610916,12.544887,184.045546,41.561712,38730.2,-9.822471,-0.016501,0.760407,-0.032395,-0.065433,-10.568595,181.528846,false,false,true,false,false,-0.029304,41.226737
176.563223,181.532308,176.563223,177.279022,57508,2020-05-12 00:00:00,"""360ONE""",12.275987,13.858216,181.992211,46.204126,40308.0,-9.80528,-0.007895,0.758314,-0.025748,-0.027763,-10.401586,178.458184,false,false,true,true,true,-0.103457,42.471084
186.522165,186.522165,178.430547,179.882889,7376,2020-05-13 00:00:00,"""360ONE""",12.599061,15.691748,181.274345,48.633562,39924.2,-9.473047,0.014688,0.748821,-0.039619,-0.01033,-10.202175,181.611867,false,false,true,true,false,-0.054835,43.70358


## Setting up the Strategies

## Stock selection

FILTERING HIGH VOLATILE AND LOW VOLUME STOCKS

In [86]:
# Most recent available values per stock
latest = df.sort('date').group_by('symbol').tail(1)
MIN_VOLUME = 300000      # stocks with avg volume < 3 lakh are removed
MAX_VOLATILITY = 0.60    # stocks with > 60% annual vol removed
filtered = latest.filter(
    (latest['sma_vol_20'] > MIN_VOLUME) &
    (latest['hv60'] < MAX_VOLATILITY)
)
# Get new filtered Dataframe
filtered_symbols = filtered['symbol'].unique().to_list()
print("Total stocks after filtering:", len(filtered_symbols))

Total stocks after filtering: 360


XG-BOOST REGRESSOR CONSTRUCTION

In [114]:

# ============================================
#  STOCK SELECTION FOR 3 STRATEGIES USING XGBOOST
# ============================================

import xgboost as xgb
import pandas as pd
import numpy as np
import polars as pl

FORWARD_DAYS = 5
TARGET_COL   = f"forward_return_{FORWARD_DAYS}d"
TRAIN_SPLIT  = "2023-01-01"
TOPN         = 20

# ----------------------------------------------------------
#  Prepare target variable (forward 5-day returns)
# ----------------------------------------------------------
df = df.with_columns(
    (pl.col("close").shift(-FORWARD_DAYS).over("symbol") / pl.col("close") - 1)
    .alias(TARGET_COL)
)

# Drop NA/inf values
for col in df.columns:
    if df[col].dtype.is_float():
        df = df.with_columns(
            pl.col(col)
            .replace(float("inf"), None)
            .replace(float("-inf"), None)
        )

df = df.drop_nulls()

# Convert to pandas for XGBoost
pdf = df.to_pandas()
pdf["date"] = pd.to_datetime(pdf["date"])

train = pdf[pdf["date"] < TRAIN_SPLIT].copy()
test  = pdf[pdf["date"] >= TRAIN_SPLIT].copy()

# ----------------------------------------------------------
#  Feature sets per strategy
# ----------------------------------------------------------

# Strategy 1 → Low Risk / Mean Reversion / Peer confirmation
features_s1 = [
    "rsi_14",
    "avg_peer_rsi14",
    "atr_14",
    "hv60",
    "daily_ret",
    "sma_vol_20",
]

# Strategy 2 → Medium Risk / Trend Continuation
features_s2 = [
    "macd",
    "macd_signal",
    "roc_14",
    "adx_14",
    "atr_14",
    "vwap_5",
    "daily_ret",
    "hv60",
]

# Strategy 3 → High Risk / VWAP Breakout Momentum
features_s3 = [
    "price_above_vwap2d",
    "rsi_14",
    "rsi_rising_2d",
    "vol_above_sma20",
    "atr_14",
    "roc_7",
    "hv60",
    "daily_ret"
]

# ----------------------------------------------------------
# Helper to train XGBoost
# ----------------------------------------------------------
def train_model(X, y):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "eta": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "n_estimators": 300,
        "n_jobs": -1,
        "random_state": 42,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X, y)
    return model

# ----------------------------------------------------------
# Train For Each Strategy
# ----------------------------------------------------------

def train_selection_model(train, test, features, pred_col):
    Xtrain = train[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    ytrain = train[TARGET_COL]

    model = train_model(Xtrain, ytrain)

    Xtest = test[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    test[pred_col] = model.predict(Xtest)

    return model, test


# Train 3 strategy models
model_s1, test = train_selection_model(train, test, features_s1, "pred_s1")
model_s2, test = train_selection_model(train, test, features_s2, "pred_s2")
model_s3, test = train_selection_model(train, test, features_s3, "pred_s3")

# ----------------------------------------------------------
# Select top N for each strategy
# ----------------------------------------------------------
def topN(test_df, pred_col, n=TOPN):
    grp = (
        test_df.groupby("symbol")[pred_col]
        .mean()
        .reset_index()
        .sort_values(pred_col, ascending=False)
        .head(n)
    )
    return grp


top_s1 = topN(test, "pred_s1", 20)
top_s2 = topN(test, "pred_s2", 20)
top_s3 = topN(test, "pred_s3", 20)

# Save
top_s1.to_csv("top20_strategy1_lowrisk.csv", index=False)
top_s2.to_csv("top20_strategy2_mediumrisk.csv", index=False)
top_s3.to_csv("top20_strategy3_highrisk.csv", index=False)

print("Strategy 1:\n", top_s1)
print("Strategy 2:\n", top_s2)
print("Strategy 3:\n", top_s3)


Strategy 1:
          symbol   pred_s1
483     YESBANK  0.021133
180        GPIL  0.009921
48         ATGL  0.009858
434       TARIL  0.009222
16   ADANIGREEN  0.009210
219        IFCI  0.008930
335      NEWGEN  0.008427
155        FACT  0.008425
129    DBREALTY  0.008424
389    RELINFRA  0.008408
185        GRSE  0.008318
174  GODFRYPHLP  0.008314
80      BLUEJET  0.007997
19     AEGISLOG  0.007952
318  MOTILALOFS  0.007912
256     JPPOWER  0.007874
275  KIRLOSBROS  0.007829
246         ITI  0.007809
486      ZENTEC  0.007763
102        CGCL  0.007670
Strategy 2:
          symbol   pred_s2
219        IFCI  0.012563
180        GPIL  0.011658
216        IDEA  0.011609
256     JPPOWER  0.011550
48         ATGL  0.011246
102        CGCL  0.010644
393      RPOWER  0.010466
113  COCHINSHIP  0.010395
15     ADANIENT  0.010379
483     YESBANK  0.009764
16   ADANIGREEN  0.009746
318  MOTILALOFS  0.009570
356       PAYTM  0.009378
155        FACT  0.009320
246         ITI  0.009312
174  GODFRYP

## Performance Assessment

## Portfolio Reallocation

## Backtesting